# Grouping and Aggregating Data

{doc}`DataFrame API Basics <01_dataframe_api_basics>` worked one row at a
time: every transformation produced one output row per input row. Most
questions a business asks are about *groups* of rows instead: revenue per
store, customers per country, best-selling product per day. This notebook
covers aggregation with `groupBy`, the subtleties of counting, pivot tables,
subtotals with `rollup` and `cube`, and window functions, which compute
group-level values without collapsing the rows.

In [1]:
from pyspark.sql import Window
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

transactions = spark.read.table("samples.bakehouse.sales_transactions")
franchises = spark.read.table("samples.bakehouse.sales_franchises")

## `groupBy` and `agg`

`groupBy` takes one or more grouping columns and returns a `GroupedData`
object. Nothing happens until you tell it how to aggregate. The shortcuts
`count`, `sum`, `avg`, `min` and `max` cover the simple cases:

In [2]:
transactions.groupBy("paymentMethod").count().show()

+-------------+-----+
|paymentMethod|count|
+-------------+-----+
|         amex| 1106|
|   mastercard| 1144|
|         visa| 1083|
+-------------+-----+



For anything more, use `agg` with functions from `pyspark.sql.functions`.
Give every aggregate a name with `alias`, otherwise you get column names
such as `sum(totalPrice)`:

In [3]:
product_summary = (
    transactions
    .groupBy("product")
    .agg(
        F.count("*").alias("transactions"),
        F.sum("quantity").alias("items_sold"),
        F.sum("totalPrice").alias("revenue"),
        F.round(F.avg("quantity"), 2).alias("avg_quantity"),
        F.max("quantity").alias("largest_order"),
    )
    .orderBy(F.desc("revenue"))
)

product_summary.show(truncate=False)

+----------------------+------------+----------+-------+------------+-------------+
|product               |transactions|items_sold|revenue|avg_quantity|largest_order|
+----------------------+------------+----------+-------+------------+-------------+
|Golden Gate Ginger    |586         |3865      |11595  |6.6         |60           |
|Outback Oatmeal       |561         |3733      |11199  |6.65        |60           |
|Austin Almond Biscotti|530         |3716      |11148  |7.01        |60           |
|Tokyo Tidbits         |583         |3662      |10986  |6.28        |54           |
|Pearly Pies           |550         |3595      |10785  |6.54        |60           |
|Orchard Oasis         |523         |3586      |10758  |6.86        |60           |
+----------------------+------------+----------+-------+------------+-------------+



Calling `agg` directly on a DataFrame, without `groupBy`, aggregates the
whole DataFrame into a single row:

In [4]:
transactions.agg(
    F.count("*").alias("transactions"),
    F.sum("totalPrice").alias("revenue"),
    F.min("dateTime").alias("first_sale"),
    F.max("dateTime").alias("last_sale"),
).show(truncate=False)

+------------+-------+--------------------------+--------------------------+
|transactions|revenue|first_sale                |last_sale                 |
+------------+-------+--------------------------+--------------------------+
|3333        |66471  |2024-05-01 00:03:28.344552|2024-05-17 11:48:38.084275|
+------------+-------+--------------------------+--------------------------+



:::{admonition} Note: Aggregations Shuffle, But Only a Little
:class: note

`groupBy` is a wide transformation, so it triggers a shuffle, as explained
in
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`.
For aggregates like `sum`, `count`, `min` and `max`, Spark first computes a
*partial* aggregate inside each partition, so only one small row per group
per partition crosses the network. Functions that need every value of a
group at once, such as `collect_list` or an exact median, cannot be reduced
this way and are more expensive.
:::

## Counting Correctly

"How many?" has several different answers in Spark, and they differ in how
they treat missing values and duplicates. A small DataFrame makes the
difference visible:

In [5]:
visits = spark.createDataFrame(
    [
        ("Alice", "Pearly Pies"),
        ("Alice", "Pearly Pies"),
        ("Bob", "Tokyo Tidbits"),
        ("Bob", None),
        (None, "Orchard Oasis"),
    ],
    schema="customer STRING, product STRING",
)

visits.agg(
    F.count("*").alias("count_star"),
    F.count("product").alias("count_product"),
    F.count_distinct("product").alias("distinct_products"),
    F.count_distinct("customer", "product").alias("distinct_pairs"),
).show()

+----------+-------------+-----------------+--------------+
|count_star|count_product|distinct_products|distinct_pairs|
+----------+-------------+-----------------+--------------+
|         5|            4|                3|             2|
+----------+-------------+-----------------+--------------+



- `count("*")` counts **rows**, whatever they contain: 5.
- `count("product")` counts **non-null values** of `product`: 4.
- `count_distinct("product")` counts **distinct non-null values**: 3.
- `count_distinct("customer", "product")` counts distinct combinations in
  which *neither* column is null: 2, namely Alice with Pearly Pies and Bob
  with Tokyo Tidbits.

Exact distinct counts are expensive on large data, because every distinct
value has to be tracked. `approx_count_distinct` uses the HyperLogLog++
algorithm to estimate the count with a bounded error, by default about 5%,
in a fixed amount of memory:

In [6]:
transactions.agg(
    F.count_distinct("customerID").alias("exact"),
    F.approx_count_distinct("customerID").alias("approximate"),
).show()

+-----+-----------+
|exact|approximate|
+-----+-----------+
|  300|        298|
+-----+-----------+



The estimate of 298 is within 1% of the true 300 customers. On a table with
billions of rows, that small error buys a large saving in memory and time.

## Conditional Aggregation

To count or sum only *some* rows of each group, put the condition inside the
aggregate. `count_if` counts rows where a condition holds, and wrapping a
value in `F.when` without `otherwise` makes non-matching rows `NULL`, which
`sum` and `avg` ignore:

In [7]:
(
    transactions
    .groupBy("product")
    .agg(
        F.count("*").alias("transactions"),
        F.count_if(F.col("quantity") >= 30).alias("large_orders"),
        F.sum(F.when(F.col("paymentMethod") == "amex", F.col("totalPrice"))).alias("amex_revenue"),
    )
    .orderBy("product")
    .show(truncate=False)
)

+----------------------+------------+------------+------------+
|product               |transactions|large_orders|amex_revenue|
+----------------------+------------+------------+------------+
|Austin Almond Biscotti|530         |14          |3765        |
|Golden Gate Ginger    |586         |15          |4164        |
|Orchard Oasis         |523         |14          |3312        |
|Outback Oatmeal       |561         |9           |3864        |
|Pearly Pies           |550         |7           |3702        |
|Tokyo Tidbits         |583         |7           |3627        |
+----------------------+------------+------------+------------+



This computes several filtered aggregates in **one pass** over the data,
instead of one filter-then-aggregate job per condition.

## Filtering Groups

SQL filters groups with `HAVING`. In the DataFrame API you simply call
`filter` after `agg`, on the aggregated column. Here are the customers who
spent more than 300 in total:

In [8]:
big_spenders = (
    transactions
    .groupBy("customerID")
    .agg(F.sum("totalPrice").alias("total_spent"))
    .filter(F.col("total_spent") > 300)
    .orderBy(F.desc("total_spent"))
)

big_spenders.show(5)
big_spenders.count()

+----------+-----------+
|customerID|total_spent|
+----------+-----------+
|   2000112|        672|
|   2000134|        519|
|   2000248|        498|
|   2000006|        480|
|   2000275|        465|
+----------+-----------+
only showing top 5 rows


51

## Statistics and "Which Row Had the Maximum?"

Besides the basic aggregates, Spark has statistical ones such as `stddev`,
`variance`, `median`, `percentile` and its cheaper approximate cousin
`percentile_approx`. A question that comes up constantly is not "what was
the maximum?" but "*which* row had the maximum?". `max_by(value, ordering)`
returns `value` from the row where `ordering` is largest:

In [9]:
(
    transactions
    .groupBy("customerID")
    .agg(
        F.max("totalPrice").alias("largest_purchase"),
        F.max_by("product", "totalPrice").alias("largest_purchase_product"),
        F.median("totalPrice").alias("median_purchase"),
    )
    .orderBy(F.desc("largest_purchase"), "customerID")
    .show(5, truncate=False)
)

+----------+----------------+------------------------+---------------+
|customerID|largest_purchase|largest_purchase_product|median_purchase|
+----------+----------------+------------------------+---------------+
|2000023   |180             |Austin Almond Biscotti  |15.0           |
|2000067   |180             |Outback Oatmeal         |22.5           |
|2000111   |180             |Pearly Pies             |21.0           |
|2000112   |180             |Austin Almond Biscotti  |15.0           |
|2000134   |180             |Orchard Oasis           |21.0           |
+----------+----------------+------------------------+---------------+
only showing top 5 rows


If several rows tie for the maximum, `max_by` returns the value from one of
them, without guaranteeing which.

## Collecting Values into Arrays

`collect_list` gathers every value of a group into an array, and
`collect_set` does the same without duplicates. The result is a column of
type `ARRAY`, the topic of
{doc}`Working with Complex Data <04_working_with_complex_data>`:

In [10]:
(
    transactions
    .groupBy("customerID")
    .agg(F.sort_array(F.collect_set("product")).alias("products_bought"))
    .show(3, truncate=False)
)

+----------+--------------------------------------------------------------------------------------------------------+
|customerID|products_bought                                                                                         |
+----------+--------------------------------------------------------------------------------------------------------+
|2000253   |[Austin Almond Biscotti, Golden Gate Ginger, Orchard Oasis, Outback Oatmeal, Pearly Pies, Tokyo Tidbits]|
|2000226   |[Austin Almond Biscotti, Orchard Oasis, Outback Oatmeal, Pearly Pies, Tokyo Tidbits]                    |
|2000108   |[Austin Almond Biscotti, Golden Gate Ginger, Orchard Oasis, Outback Oatmeal, Pearly Pies, Tokyo Tidbits]|
+----------+--------------------------------------------------------------------------------------------------------+
only showing top 3 rows


Collecting moves all values of a group into a single row. That is fine for a
few hundred values per group, but a group with millions of values can
exhaust an executor's memory.

## Pivot Tables

A **pivot** turns the distinct values of one column into separate columns.
Group by the row dimension, `pivot` on the column dimension, then aggregate.
Here is revenue per product (rows) by payment method (columns):

In [11]:
(
    transactions
    .groupBy("product")
    .pivot("paymentMethod", ["amex", "mastercard", "visa"])
    .agg(F.sum("totalPrice"))
    .orderBy("product")
    .show(truncate=False)
)

+----------------------+----+----------+----+
|product               |amex|mastercard|visa|
+----------------------+----+----------+----+
|Austin Almond Biscotti|3765|3711      |3672|
|Golden Gate Ginger    |4164|3789      |3642|
|Orchard Oasis         |3312|3954      |3492|
|Outback Oatmeal       |3864|3771      |3564|
|Pearly Pies           |3702|3360      |3723|
|Tokyo Tidbits         |3627|3678      |3681|
+----------------------+----+----------+----+



Always pass the list of values to `pivot` when you know them. Without the
list, Spark first runs an extra job to find the distinct values, and the
columns of your output change whenever the data does.

The reverse operation, `unpivot` (alias `melt`), turns columns back into
rows. It needs the identifier columns, the columns to unpivot, and names for
the two new columns. Unpivoting the Pearly Pies row of the table above gives
back one row per payment method:

In [12]:
wide = spark.createDataFrame(
    [("Pearly Pies", 3702, 3360, 3723)],
    schema="product STRING, amex BIGINT, mastercard BIGINT, visa BIGINT",
)

wide.unpivot(
    ids=["product"],
    values=["amex", "mastercard", "visa"],
    variableColumnName="payment_method",
    valueColumnName="revenue",
).show()

+-----------+--------------+-------+
|    product|payment_method|revenue|
+-----------+--------------+-------+
|Pearly Pies|          amex|   3702|
|Pearly Pies|    mastercard|   3360|
|Pearly Pies|          visa|   3723|
+-----------+--------------+-------+



## Subtotals with `rollup` and `cube`

Reports often need subtotals: revenue per city, per country, and in total.
`rollup(a, b)` groups by `(a, b)`, then by `(a)`, then by nothing, all in
one query. In the subtotal rows, the columns that were rolled up are `NULL`.
To tell those `NULL`s apart from real missing values, use `grouping_id`,
whose bits mark which columns were aggregated away:

In [13]:
sales_by_location = transactions.join(franchises, "franchiseID")

(
    sales_by_location
    .filter(F.col("country").isin("Australia", "Japan"))
    .rollup("country", "city")
    .agg(
        F.sum("totalPrice").alias("revenue"),
        F.grouping_id().alias("level"),
    )
    .orderBy("country", "city")
    .show(truncate=False)
)

+---------+----------+-------+-----+
|country  |city      |revenue|level|
+---------+----------+-------+-----+
|NULL     |NULL      |29802  |3    |
|Australia|NULL      |8211   |1    |
|Australia|Adelaide  |840    |0    |
|Australia|Brisbane  |1275   |0    |
|Australia|Gold Coast|1176   |0    |
|Australia|Melbourne |1008   |0    |
|Australia|Perth     |1122   |0    |
|Australia|Sydney    |2790   |0    |
|Japan    |NULL      |21591  |1    |
|Japan    |Aomori    |1392   |0    |
|Japan    |Chiba     |1482   |0    |
|Japan    |Fukuoka   |1212   |0    |
|Japan    |Hiratsuka |1020   |0    |
|Japan    |Hiroshima |834    |0    |
|Japan    |Kagoshima |1161   |0    |
|Japan    |Kanazawa  |1353   |0    |
|Japan    |Kobe      |1281   |0    |
|Japan    |Kumamoto  |915    |0    |
|Japan    |Kyoto     |1212   |0    |
|Japan    |Matsuyama |1338   |0    |
+---------+----------+-------+-----+
only showing top 20 rows


Rows with `level` 0 are cities, rows with `level` 1 are country subtotals
(only `city` rolled up), and the single row with `level` 3 (binary `11`,
both columns rolled up) is the grand total. `cube(a, b)` goes further and
produces every combination of grouping columns, including `(b)` on its own,
which is useful when the dimensions are not hierarchical. (The `join` used to
attach the country is covered in
{doc}`DataFrame Relational Operations <03_dataframe_relational_operations>`.)

## Window Functions

An aggregate collapses each group into one row. Often you want the group
value *next to* each row instead: each sale's share of its store's revenue,
the rank of a product within its store, the previous day's revenue. These
are **window functions**. A window is defined with a `Window` spec:

- `partitionBy(...)`: which rows belong to the same group, like `groupBy`.
- `orderBy(...)`: the order of rows within the group, needed for ranking and
  running calculations.
- `rowsBetween(start, end)` or `rangeBetween(start, end)`: the **frame**,
  meaning which rows around the current one the function sees.

The function is applied with `.over(window)`, and the result is a new
column. No rows are removed.

### Group Totals Next to Each Row

A window with only `partitionBy` makes every row see its whole group:

In [14]:
by_product = Window.partitionBy("product")

(
    transactions
    .select(
        "transactionID",
        "product",
        "totalPrice",
        F.sum("totalPrice").over(by_product).alias("product_revenue"),
    )
    .withColumn("share_pct", F.round(100 * F.col("totalPrice") / F.col("product_revenue"), 3))
    .orderBy(F.desc("share_pct"))
    .show(5, truncate=False)
)

+-------------+----------------------+----------+---------------+---------+
|transactionID|product               |totalPrice|product_revenue|share_pct|
+-------------+----------------------+----------+---------------+---------+
|1000235      |Orchard Oasis         |180       |10758          |1.673    |
|1002518      |Orchard Oasis         |180       |10758          |1.673    |
|1001783      |Pearly Pies           |180       |10785          |1.669    |
|1001228      |Austin Almond Biscotti|180       |11148          |1.615    |
|1000843      |Austin Almond Biscotti|180       |11148          |1.615    |
+-------------+----------------------+----------+---------------+---------+
only showing top 5 rows


### Ranking

`row_number`, `rank` and `dense_rank` number the rows in each partition in
the window's order. They differ only in how they handle ties, so a tiny
example with a tie shows the difference best.

:::::{dropdown} Example: Three Ways to Rank Tied Values
:open:
:color: primary
:icon: pencil
:class-body: example-body

Four stores sold 50, 40, 40 and 30 items. How does each function rank them,
from most to fewest items?

**`row_number`** gives every row a distinct number. The two stores tied on
40 get 2 and 3, in an order Spark does not guarantee:

```text
50 -> 1,  40 -> 2,  40 -> 3,  30 -> 4
```

**`rank`** gives tied rows the same number and then *skips*, like sports
results, so the store after the tie is 4th:

```text
50 -> 1,  40 -> 2,  40 -> 2,  30 -> 4
```

**`dense_rank`** gives tied rows the same number and does *not* skip:

```text
50 -> 1,  40 -> 2,  40 -> 2,  30 -> 3
```

To pick exactly *n* rows per group, use `row_number`. To pick "everything in
the top *n* positions, ties included", use `rank` or `dense_rank`.
:::::

Checking the example against Spark:

In [15]:
stores = spark.createDataFrame(
    [("A", 50), ("B", 40), ("C", 40), ("D", 30)],
    schema="store STRING, items INT",
)

most_items_first = Window.orderBy(F.desc("items"))

stores.select(
    "store",
    "items",
    F.row_number().over(most_items_first).alias("row_number"),
    F.rank().over(most_items_first).alias("rank"),
    F.dense_rank().over(most_items_first).alias("dense_rank"),
).show()

+-----+-----+----------+----+----------+
|store|items|row_number|rank|dense_rank|
+-----+-----+----------+----+----------+
|    A|   50|         1|   1|         1|
|    B|   40|         2|   2|         2|
|    C|   40|         3|   2|         2|
|    D|   30|         4|   4|         3|
+-----+-----+----------+----+----------+



This window has no `partitionBy`, so Spark has to move every row into a
single partition to number them, and it logs a warning saying so. That is
fine for four rows, but on a large
DataFrame it funnels all the data through one task. Always partition a
window on real data.

The classic use of ranking is **top-n per group**. Here are the two
best-selling products, by revenue, in each of the first three franchises:

In [16]:
franchise_product_revenue = (
    transactions
    .groupBy("franchiseID", "product")
    .agg(F.sum("totalPrice").alias("revenue"))
)

best_first = Window.partitionBy("franchiseID").orderBy(F.desc("revenue"), "product")

(
    franchise_product_revenue
    .withColumn("position", F.row_number().over(best_first))
    .filter(F.col("position") <= 2)
    .orderBy("franchiseID", "position")
    .show(6)
)

+-----------+--------------------+-------+--------+
|franchiseID|             product|revenue|position|
+-----------+--------------------+-------+--------+
|    3000000|     Outback Oatmeal|    570|       1|
|    3000000|       Tokyo Tidbits|    558|       2|
|    3000001|         Pearly Pies|    222|       1|
|    3000001|     Outback Oatmeal|    216|       2|
|    3000002|Austin Almond Bis...|    558|       1|
|    3000002|       Tokyo Tidbits|    534|       2|
+-----------+--------------------+-------+--------+
only showing top 6 rows


Adding `product` as a second sort key breaks ties deterministically, so the
result is the same on every run.

### Running Totals and Moving Averages

With an `orderBy`, a window function sees a **frame** of rows relative to
the current one. `rowsBetween(Window.unboundedPreceding, Window.currentRow)`
means "from the start of the partition up to this row", which gives a
running total. `rowsBetween(-2, 0)` means "this row and the two before it",
a moving window of three rows. `lag` and `lead` read the value from an
earlier or later row.

In [17]:
daily = (
    transactions
    .groupBy(F.to_date("dateTime").alias("sale_date"))
    .agg(F.sum("totalPrice").alias("revenue"))
)

by_date = Window.orderBy("sale_date")

daily_trend = daily.select(
    "sale_date",
    "revenue",
    F.sum("revenue").over(by_date.rowsBetween(Window.unboundedPreceding, Window.currentRow)).alias("running_total"),
    F.round(F.avg("revenue").over(by_date.rowsBetween(-2, Window.currentRow)), 1).alias("moving_avg_3d"),
    (F.col("revenue") - F.lag("revenue", 1).over(by_date)).alias("change_vs_previous_day"),
).orderBy("sale_date")

daily_trend.show(20)

+----------+-------+-------------+-------------+----------------------+
| sale_date|revenue|running_total|moving_avg_3d|change_vs_previous_day|
+----------+-------+-------------+-------------+----------------------+
|2024-05-01|   4128|         4128|       4128.0|                  NULL|
|2024-05-02|   4074|         8202|       4101.0|                   -54|
|2024-05-03|   4278|        12480|       4160.0|                   204|
|2024-05-04|   3822|        16302|       4058.0|                  -456|
|2024-05-05|   3945|        20247|       4015.0|                   123|
|2024-05-06|   4500|        24747|       4089.0|                   555|
|2024-05-07|   3894|        28641|       4113.0|                  -606|
|2024-05-08|   3921|        32562|       4105.0|                    27|
|2024-05-09|   4320|        36882|       4045.0|                   399|
|2024-05-10|   3729|        40611|       3990.0|                  -591|
|2024-05-11|   3747|        44358|       3932.0|                

:::::{dropdown} Example: Checking the Running Total and the Moving Average
:open:
:color: primary
:icon: pencil
:class-body: example-body

Take the first three days of the output above, with revenues of
4128, 4074 and 4278.

**Running total on day three.** The frame runs from the first row up to the
current one, so it adds all three days:

$$
4128 + 4074 + 4278 = 12480
$$

**Three-day moving average on day three.** The frame `rowsBetween(-2, 0)`
covers the same three rows, so the average is:

$$
\frac{12480}{3} = 4160
$$

**Moving average on day one.** There are no earlier rows, so the frame
holds just one row and the "average" is that day's revenue, 4128.

Both values match the `running_total` and `moving_avg_3d` columns of the
output. The first row of `change_vs_previous_day` is `NULL`, because `lag`
has no previous row to read.
:::::

:::{admonition} Note: The Default Frame Is a Range, Not Rows
:class: note

If a window has an `orderBy` but you do not specify a frame, Spark uses
`rangeBetween(Window.unboundedPreceding, Window.currentRow)`. A *range*
frame includes all rows whose ordering value equals the current row's, so
rows that tie on the ordering column all receive the same running total.
Specify `rowsBetween` explicitly whenever you need a strict row-by-row
running total.
:::

The running total over the whole period must equal the overall revenue,
which makes a cheap sanity check:

In [18]:
daily_trend.orderBy(F.desc("sale_date")).first()["running_total"] == transactions.agg(F.sum("totalPrice")).first()[0]

True

Next, {doc}`DataFrame Relational Operations <03_dataframe_relational_operations>`
combines DataFrames with joins and set operations.

## Resources

- [Aggregate functions (Apache Spark)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#aggregate-functions)
- [Window functions (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html)
- [GROUP BY clause, including ROLLUP and CUBE (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-groupby.html)
- [PIVOT clause (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-pivot.html)